In [61]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [62]:
from pathlib import Path
import pandas as pd
import sys

In [63]:
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT = PROJECT_ROOT / "data/sample"

SAMPLE_ID = "44b6_0113de3b"

PROCESSED_DIR = (
        DATA_ROOT
        / "processed"
        / "stage_6_processed_dataset"
        / SAMPLE_ID
)

CELLS_DIR = PROCESSED_DIR / "cells"

In [64]:
cell_files = sorted(CELLS_DIR.glob("t*.csv"))

time_frames = [
    pd.read_csv(file)
    for file in cell_files
]

print(f"Loaded {len(time_frames)} timepoints.")

Loaded 20 timepoints.


In [65]:
import pandas as pd
import json
from pathlib import Path

output_dir = Path("../data/sample/processed/stage_8_track_stitching")  # Change this to your output directory

# ------------------------------------------------------------
# Load detections
# ------------------------------------------------------------
detections_df = pd.read_csv(output_dir / "detections.csv")

# If you want the original list of DataFrames (time_frames)
time_frames = [
    frame_df.reset_index(drop=True)
    for _, frame_df in detections_df.groupby("frame")
]

# ------------------------------------------------------------
# Load tracks
# ------------------------------------------------------------
tracks = pd.read_csv(output_dir / "tracks.csv")

# ------------------------------------------------------------
# Load segmentation events
# ------------------------------------------------------------
segmentation_events = pd.read_csv(output_dir / "segmentation_events.csv")

# ------------------------------------------------------------
# Load track endings
# ------------------------------------------------------------
track_endings = pd.read_csv(output_dir / "track_endings.csv")

# ------------------------------------------------------------
# Load metadata
# ------------------------------------------------------------
with open(output_dir / "metadata.json", "r") as f:
    metadata = json.load(f)

print("Loaded all Stage 7 results.")

Loaded all Stage 7 results.


Approximately assign cell id

In [66]:
import numpy as np
from scipy.spatial import cKDTree

# Reload the original per-frame cell files (these have cell_id + centroids)
cell_files = sorted(CELLS_DIR.glob("t*.csv"))

cell_frames = []
for i, file in enumerate(cell_files):
    df = pd.read_csv(file)
    df["frame"] = i  # verify this matches the frame numbering used in tracks.csv
    cell_frames.append(df)

cells_df = pd.concat(cell_frames, ignore_index=True)

# Nearest-neighbor match each track row to its source cell, per frame
tracks = tracks.copy()
tracks["cell_id"] = -1

for frame, frame_tracks in tracks.groupby("frame"):
    frame_cells = cells_df[cells_df["frame"] == frame]
    if frame_cells.empty:
        continue
    tree = cKDTree(frame_cells[["centroid_z", "centroid_y", "centroid_x"]].to_numpy())
    dist, idx = tree.query(frame_tracks[["z", "y", "x"]].to_numpy())
    tracks.loc[frame_tracks.index, "cell_id"] = frame_cells["cell_id"].to_numpy()[idx]

# sanity check — distances should be ~0 (or very small) if the match is correct
print("max match distance:", dist.max() if len(dist) else "n/a")
print("unmatched rows:", (tracks["cell_id"] == -1).sum())

max match distance: 5.684341886080802e-14
unmatched rows: 0


In [67]:
import zarr

ZARR_PATH = (
        DATA_ROOT
        / "biohub_5samples_20timepoints"
        / "train"
        / SAMPLE_ID
        / f"{SAMPLE_ID}.zarr"
)

ARRAY_PATH = ZARR_PATH / "0"

original_volume = zarr.open_array(
    str(ARRAY_PATH),
    mode="r"
)

In [68]:
# ------------------------------------------------------------
# Convert tracks to Napari format
# ------------------------------------------------------------

tracks_array = tracks[
    ["track_id", "frame", "z", "y", "x"]
].to_numpy(dtype=float)

points_array = tracks[
    ["frame", "z", "y", "x"]
].to_numpy(dtype=float)

track_ids = tracks["track_id"].to_numpy()

Load pre-processed data

In [69]:
from pathlib import Path

import dask.array as da
import numpy as np


PREPROCESSING_DIR = PROCESSED_DIR / "preprocessing"
MASKING_DIR = PROCESSED_DIR / "masking"
SEGMENTATION_DIR = PROCESSED_DIR / "segmentation"


def load_npy_time_series(
        directory: Path,
        expected_frames: int | None = None,
) -> tuple[da.Array, list[Path]]:
    """
    Load per-frame 3D .npy files as one lazy 4D Dask array.

    Expected files:
        t000.npy
        t001.npy
        ...

    Returned shape:
        (T, Z, Y, X)
    """

    files = sorted(directory.glob("t*.npy"))

    if not files:
        raise FileNotFoundError(
            f"No time-point arrays found in:\n{directory}"
        )

    if expected_frames is not None and len(files) != expected_frames:
        raise ValueError(
            f"{directory.name}: expected {expected_frames} frames, "
            f"but found {len(files)}."
        )

    first = np.load(
        files[0],
        mmap_mode="r",
        allow_pickle=False,
    )

    if first.ndim != 3:
        raise ValueError(
            f"Expected 3D arrays in {directory}, "
            f"but {files[0].name} has shape {first.shape}."
        )

    spatial_shape = first.shape

    # Small chunks allow the extractor to read only the selected crop.
    chunks = (
        min(8, spatial_shape[0]),
        min(64, spatial_shape[1]),
        min(64, spatial_shape[2]),
    )

    arrays = []

    for path in files:
        array = np.load(
            path,
            mmap_mode="r",
            allow_pickle=False,
        )

        if array.shape != spatial_shape:
            raise ValueError(
                f"Inconsistent shape in {path.name}: "
                f"expected {spatial_shape}, found {array.shape}."
            )

        arrays.append(
            da.from_array(
                array,
                chunks=chunks,
            )
        )

    sequence = da.stack(
        arrays,
        axis=0,
    )

    return sequence, files

In [70]:
NUM_TIMEPOINTS = original_volume.shape[0]

preprocessed_volume, preprocessing_files = load_npy_time_series(
    PREPROCESSING_DIR,
    expected_frames=NUM_TIMEPOINTS,
)

binary_mask_volume, masking_files = load_npy_time_series(
    MASKING_DIR,
    expected_frames=NUM_TIMEPOINTS,
)

instance_labels_volume, segmentation_files = load_npy_time_series(
    SEGMENTATION_DIR,
    expected_frames=NUM_TIMEPOINTS,
)

In [71]:
print(
    "Raw:",
    original_volume.shape,
    original_volume.dtype,
)

print(
    "Preprocessed:",
    preprocessed_volume.shape,
    preprocessed_volume.dtype,
)

print(
    "Binary mask:",
    binary_mask_volume.shape,
    binary_mask_volume.dtype,
)

print(
    "Instance labels:",
    instance_labels_volume.shape,
    instance_labels_volume.dtype,
)

Raw: (20, 64, 256, 256) uint16
Preprocessed: (20, 64, 256, 256) float32
Binary mask: (20, 64, 256, 256) bool
Instance labels: (20, 64, 256, 256) int32


**Broken Tracks**

Red cells -> Tracks that will break before the last frame

Green cells -> Tracks that will born after the first frame

In [72]:
from diagnostics.cell_volume_extraction.napari_extractor import (
    add_cell_volume_extractor,
)

In [73]:
# Endpoint cells this close to any spatial boundary are treated
# as entering/leaving the imaging volume rather than tracking failures.
BOUNDARY_MARGIN_UM = 4.0

# Whether boundary-entry and boundary-exit tracks should be added
# to Napari as separate diagnostic layers.
SHOW_BOUNDARY_TRACKS = True

In [74]:
import napari
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

VOXEL_SIZE = (
    1.625,      # Z
    0.40625,    # Y
    0.40625,    # X
)

SPATIAL_SHAPE_ZYX = original_volume.shape[-3:]


# ------------------------------------------------------------
# Find new and ended tracks
# ------------------------------------------------------------

first_frame = tracks["frame"].min()
last_frame = tracks["frame"].max()

track_summary = (
    tracks.groupby("track_id")
    .agg(
        first_frame=("frame", "min"),
        last_frame=("frame", "max"),
    )
    .reset_index()
)

# All apparent births and deaths before boundary filtering
all_new_track_ids = track_summary.loc[
    track_summary["first_frame"] > first_frame,
    "track_id",
]

all_ended_track_ids = track_summary.loc[
    track_summary["last_frame"] < last_frame,
    "track_id",
]


# ------------------------------------------------------------
# Endpoint extraction
# ------------------------------------------------------------

def get_track_endpoints(
        tracks_df,
        track_ids,
        endpoint,
):
    """
    Return the first or last detection of each selected track.

    Parameters
    ----------
    tracks_df : pandas.DataFrame
        Complete tracks DataFrame.

    track_ids : iterable
        IDs of the tracks whose endpoints should be extracted.

    endpoint : {"start", "end"}
        Which endpoint to extract.
    """

    selected = tracks_df[
        tracks_df["track_id"].isin(track_ids)
    ]

    if selected.empty:
        return selected.copy()

    grouped_frames = selected.groupby("track_id")["frame"]

    if endpoint == "start":
        row_indices = grouped_frames.idxmin()

    elif endpoint == "end":
        row_indices = grouped_frames.idxmax()

    else:
        raise ValueError(
            "endpoint must be either 'start' or 'end'"
        )

    return (
        selected.loc[row_indices]
        .sort_values("track_id")
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# Boundary classification
# ------------------------------------------------------------

def classify_boundary_endpoints(
        endpoint_rows,
        cells_df,
        spatial_shape_zyx,
        voxel_size_zyx,
        boundary_margin_um,
):
    """
    Determine whether each track endpoint is close to a volume boundary.

    The endpoint cell bounding box is used when available. If bounding-box
    information is unavailable, the endpoint centroid is used instead.
    """

    result = endpoint_rows.copy()

    if result.empty:
        result["boundary_distance_um"] = pd.Series(dtype=float)
        result["is_boundary_endpoint"] = pd.Series(dtype=bool)
        return result

    voxel_size = np.asarray(
        voxel_size_zyx,
        dtype=float,
    )

    spatial_shape = np.asarray(
        spatial_shape_zyx,
        dtype=float,
    )

    # --------------------------------------------------------
    # Centroid-based boundary distance
    # --------------------------------------------------------

    coordinates_zyx = result[
        ["z", "y", "x"]
    ].to_numpy(dtype=float)

    distance_to_lower_faces_um = (
            coordinates_zyx * voxel_size
    )

    distance_to_upper_faces_um = (
                                         spatial_shape - 1 - coordinates_zyx
                                 ) * voxel_size

    centroid_boundary_distance_um = np.minimum(
        distance_to_lower_faces_um,
        distance_to_upper_faces_um,
    ).min(axis=1)

    # --------------------------------------------------------
    # Prefer cell bounding-box distance when available
    # --------------------------------------------------------

    bbox_columns = [
        "z_min",
        "y_min",
        "x_min",
        "z_max",
        "y_max",
        "x_max",
    ]

    bbox_available = (
            "cell_id" in result.columns
            and "frame" in cells_df.columns
            and "cell_id" in cells_df.columns
            and set(bbox_columns).issubset(cells_df.columns)
    )

    if bbox_available:
        cell_lookup = (
            cells_df[
                [
                    "frame",
                    "cell_id",
                    *bbox_columns,
                ]
            ]
            .drop_duplicates(
                subset=["frame", "cell_id"]
            )
        )

        result = result.merge(
            cell_lookup,
            on=["frame", "cell_id"],
            how="left",
        )

        bbox_min_zyx = result[
            ["z_min", "y_min", "x_min"]
        ].to_numpy(dtype=float)

        bbox_max_zyx = result[
            ["z_max", "y_max", "x_max"]
        ].to_numpy(dtype=float)

        bbox_distance_to_lower_faces_um = (
                bbox_min_zyx * voxel_size
        )

        # z_max, y_max and x_max are treated as exclusive bounds.
        bbox_distance_to_upper_faces_um = (
                                                  spatial_shape - bbox_max_zyx
                                          ) * voxel_size

        bbox_boundary_distance_um = np.minimum(
            bbox_distance_to_lower_faces_um,
            bbox_distance_to_upper_faces_um,
        ).min(axis=1)

        valid_bbox = np.isfinite(
            bbox_boundary_distance_um
        )

        result["boundary_distance_um"] = np.where(
            valid_bbox,
            bbox_boundary_distance_um,
            centroid_boundary_distance_um,
        )

    else:
        result["boundary_distance_um"] = (
            centroid_boundary_distance_um
        )

    result["is_boundary_endpoint"] = (
            result["boundary_distance_um"]
            <= boundary_margin_um
    )

    return result


# ------------------------------------------------------------
# Extract relevant endpoints
# ------------------------------------------------------------

new_track_endpoints = get_track_endpoints(
    tracks_df=tracks,
    track_ids=all_new_track_ids,
    endpoint="start",
)

ended_track_endpoints = get_track_endpoints(
    tracks_df=tracks,
    track_ids=all_ended_track_ids,
    endpoint="end",
)


# ------------------------------------------------------------
# Classify endpoints as boundary or non-boundary
# ------------------------------------------------------------

new_track_endpoints = classify_boundary_endpoints(
    endpoint_rows=new_track_endpoints,
    cells_df=cells_df,
    spatial_shape_zyx=SPATIAL_SHAPE_ZYX,
    voxel_size_zyx=VOXEL_SIZE,
    boundary_margin_um=BOUNDARY_MARGIN_UM,
)

ended_track_endpoints = classify_boundary_endpoints(
    endpoint_rows=ended_track_endpoints,
    cells_df=cells_df,
    spatial_shape_zyx=SPATIAL_SHAPE_ZYX,
    voxel_size_zyx=VOXEL_SIZE,
    boundary_margin_um=BOUNDARY_MARGIN_UM,
)


# ------------------------------------------------------------
# Separate failure candidates from boundary entries/exits
# ------------------------------------------------------------

boundary_new_track_ids = new_track_endpoints.loc[
    new_track_endpoints["is_boundary_endpoint"],
    "track_id",
]

boundary_ended_track_ids = ended_track_endpoints.loc[
    ended_track_endpoints["is_boundary_endpoint"],
    "track_id",
]

new_track_ids = new_track_endpoints.loc[
    ~new_track_endpoints["is_boundary_endpoint"],
    "track_id",
]

ended_track_ids = ended_track_endpoints.loc[
    ~ended_track_endpoints["is_boundary_endpoint"],
    "track_id",
]


# ------------------------------------------------------------
# Build track DataFrames
# ------------------------------------------------------------

# Non-boundary failure candidates
new_tracks = tracks[
    tracks["track_id"].isin(new_track_ids)
].copy()

ended_tracks = tracks[
    tracks["track_id"].isin(ended_track_ids)
].copy()

# Boundary entries and exits
boundary_new_tracks = tracks[
    tracks["track_id"].isin(boundary_new_track_ids)
].copy()

boundary_ended_tracks = tracks[
    tracks["track_id"].isin(boundary_ended_track_ids)
].copy()


# ------------------------------------------------------------
# Launch Napari
# ------------------------------------------------------------

viewer = napari.Viewer(ndisplay=3)


# ------------------------------------------------------------
# Failure-analysis layers
# ------------------------------------------------------------

viewer.add_image(
    original_volume,
    name="Raw Volume",
    scale=(1, *VOXEL_SIZE),
    rendering="mip",
    colormap="gray",
    contrast_limits=[
        np.percentile(original_volume, 1),
        np.percentile(original_volume, 99.8),
    ],
)


# ------------------------------------------------------------
# Ended failure candidates
# ------------------------------------------------------------

if not ended_tracks.empty:
    viewer.add_tracks(
        ended_tracks[
            ["track_id", "frame", "z", "y", "x"]
        ].to_numpy(float),
        name="Ended Tracks",
        scale=(1, *VOXEL_SIZE),
        tail_length=20,
    )

    viewer.add_points(
        ended_tracks[
            ["frame", "z", "y", "x"]
        ].to_numpy(float),
        name="Ended Centroids",
        scale=(1, *VOXEL_SIZE),
        size=4,
        face_color="red",
    )


# ------------------------------------------------------------
# New failure candidates
# ------------------------------------------------------------

if not new_tracks.empty:
    viewer.add_tracks(
        new_tracks[
            ["track_id", "frame", "z", "y", "x"]
        ].to_numpy(float),
        name="New Tracks",
        scale=(1, *VOXEL_SIZE),
        tail_length=20,
    )

    viewer.add_points(
        new_tracks[
            ["frame", "z", "y", "x"]
        ].to_numpy(float),
        name="New Centroids",
        scale=(1, *VOXEL_SIZE),
        size=4,
        face_color="lime",
    )


# ------------------------------------------------------------
# Optional boundary tracks
# ------------------------------------------------------------

if SHOW_BOUNDARY_TRACKS:

    # Tracks entering through a volume boundary
    if not boundary_new_tracks.empty:
        viewer.add_tracks(
            boundary_new_tracks[
                ["track_id", "frame", "z", "y", "x"]
            ].to_numpy(float),
            name="Boundary Entry Tracks",
            scale=(1, *VOXEL_SIZE),
            tail_length=20,
        )

        viewer.add_points(
            boundary_new_tracks[
                ["frame", "z", "y", "x"]
            ].to_numpy(float),
            name="Boundary Entry Centroids",
            scale=(1, *VOXEL_SIZE),
            size=4,
            face_color="cyan",
        )

    # Tracks leaving through a volume boundary
    if not boundary_ended_tracks.empty:
        viewer.add_tracks(
            boundary_ended_tracks[
                ["track_id", "frame", "z", "y", "x"]
            ].to_numpy(float),
            name="Boundary Exit Tracks",
            scale=(1, *VOXEL_SIZE),
            tail_length=20,
        )

        viewer.add_points(
            boundary_ended_tracks[
                ["frame", "z", "y", "x"]
            ].to_numpy(float),
            name="Boundary Exit Centroids",
            scale=(1, *VOXEL_SIZE),
            size=4,
            face_color="orange",
        )


# ------------------------------------------------------------
# All original tracks
# ------------------------------------------------------------

viewer.add_image(
    original_volume,
    name="Raw Volume - all",
    rendering="mip",
    colormap="gray",
    contrast_limits=[
        np.percentile(original_volume, 1),
        np.percentile(original_volume, 99.8),
    ],
    scale=(1, *VOXEL_SIZE),
).visible = False

viewer.add_tracks(
    tracks_array,
    name="Tracks - all",
    tail_length=20,
    scale=(1, *VOXEL_SIZE),
).visible = False

viewer.add_points(
    points_array,
    name="Centroids - all",
    size=4,
    face_color="red",
    properties={
        "cell_id": tracks["cell_id"].to_numpy(),
        "track_id": track_ids,
    },
    scale=(1, *VOXEL_SIZE),
    text={
        "string": "{cell_id}",
        "size": 8,
        "color": "white",
        "anchor": "center",
    },
).visible = False


# ------------------------------------------------------------
# Diagnostic summary
# ------------------------------------------------------------

print(
    f"New tracks before boundary filtering: "
    f"{len(all_new_track_ids)}"
)

print(
    f"Boundary-entry tracks: "
    f"{len(boundary_new_track_ids)}"
)

print(
    f"New failure candidates: "
    f"{len(new_track_ids)}"
)

print()

print(
    f"Ended tracks before boundary filtering: "
    f"{len(all_ended_track_ids)}"
)

print(
    f"Boundary-exit tracks: "
    f"{len(boundary_ended_track_ids)}"
)

print(
    f"Ended failure candidates: "
    f"{len(ended_track_ids)}"
)

print()

print(
    f"SHOW_BOUNDARY_TRACKS: "
    f"{SHOW_BOUNDARY_TRACKS}"
)


# ------------------------------------------------------------
# Cell-volume extraction tool
# ------------------------------------------------------------

cell_extractor = add_cell_volume_extractor(
    viewer=viewer,
    cells=cells_df,

    # Required
    image_volume=original_volume,

    # Diagnostic pipeline outputs
    binary_mask_volume=binary_mask_volume,
    instance_labels_volume=instance_labels_volume,

    # Include when available
    preprocessed_volume=preprocessed_volume,

    sample_id=SAMPLE_ID,
    voxel_size_zyx=VOXEL_SIZE,
    default_box_size=(12, 50, 50),

    source_cells_dir=CELLS_DIR,
    source_zarr_array=ARRAY_PATH,

    extract_key="E",
)

napari.run()

New tracks before boundary filtering: 458
Boundary-entry tracks: 180
New failure candidates: 278

Ended tracks before boundary filtering: 436
Boundary-exit tracks: 168
Ended failure candidates: 268

SHOW_BOUNDARY_TRACKS: True
